# Phase 1: Market Data and Returns

## Project context

This notebook starts the Portfolio Optimization and Factor Research Lab by building the market data foundation for later CAPM, factor research, portfolio optimization, and backtesting work. Phase 1 focuses only on price loading, return calculation, risk and return summary, and benchmark comparison.

## Ticker universe and benchmark

- Asset universe: AAPL, MSFT, JPM, PG, XOM, JNJ, KO, NVDA
- Benchmark: SPY
- Period: 2019-01-01 to latest available yfinance data
- Data source: Yahoo Finance via `yfinance`

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.data_loader import download_price_data, extract_adjusted_close, save_price_data
from src.returns import (
    calculate_daily_returns,
    calculate_monthly_returns,
    calculate_max_drawdown,
    summarize_asset_performance,
)
from src.visualization import (
    plot_price_history,
    plot_cumulative_returns,
    plot_correlation_heatmap,
    plot_risk_return_scatter,
)

TICKERS = ["AAPL", "MSFT", "JPM", "PG", "XOM", "JNJ", "KO", "NVDA"]
BENCHMARK = "SPY"
ALL_TICKERS = TICKERS + [BENCHMARK]
START_DATE = "2019-01-01"

RETURNS_DIR = OUTPUTS_DIR / "returns"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RETURNS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    """Save a Plotly figure as PNG, with a data-driven Pillow fallback if Kaleido is unavailable."""
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    title_font = ImageFont.load_default()
    text_font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=title_font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=text_font)

    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "lines":
        frame = data.dropna(how="all")
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#17becf"]
        for idx, col in enumerate(frame.columns):
            series = frame[col].astype(float)
            y_norm = _normalize(series.values)
            points = []
            for i, value in enumerate(y_norm):
                x = left + (right - left) * i / max(len(y_norm) - 1, 1)
                y = bottom - (bottom - top) * value
                points.append((x, y))
            if len(points) > 1:
                draw.line(points, fill=colors[idx % len(colors)], width=2)
            draw.text((right - 130, top + 18 * idx), str(col), fill=colors[idx % len(colors)], font=text_font)
    elif chart_type == "heatmap":
        matrix = data.astype(float)
        labels = list(matrix.columns)
        n = len(labels)
        cell = min((right - left) / max(n, 1), (bottom - top) / max(n, 1))
        for i, row in enumerate(labels):
            for j, col in enumerate(labels):
                value = matrix.loc[row, col]
                red = int(255 * max(value, 0))
                blue = int(255 * abs(min(value, 0)))
                green = int(230 * (1 - abs(value)))
                x0 = left + j * cell
                y0 = top + i * cell
                draw.rectangle((x0, y0, x0 + cell, y0 + cell), fill=(red, green, blue), outline="white")
                draw.text((x0 + 4, y0 + 4), f"{value:.2f}", fill="black", font=text_font)
        for i, label in enumerate(labels):
            draw.text((left + i * cell + 4, top - 18), label, fill="black", font=text_font)
            draw.text((left - 50, top + i * cell + 4), label, fill="black", font=text_font)
    elif chart_type == "scatter":
        frame = data.copy()
        x = _normalize(frame["annualized_volatility"].values)
        y = _normalize(frame["annualized_return"].values)
        for i, ticker in enumerate(frame.index):
            px = left + (right - left) * x[i]
            py = bottom - (bottom - top) * y[i]
            draw.ellipse((px - 5, py - 5, px + 5, py + 5), fill="#1f77b4")
            draw.text((px + 8, py - 8), str(ticker), fill="black", font=text_font)

    image.save(output_path)


## Market data download

In [3]:
raw_data = download_price_data(ALL_TICKERS, start_date=START_DATE)
prices = extract_adjusted_close(raw_data)
prices = prices.reindex(columns=ALL_TICKERS)
prices = prices.dropna(how="all")

missing_tickers = [ticker for ticker in ALL_TICKERS if ticker not in prices.columns or prices[ticker].dropna().empty]
if missing_tickers:
    raise ValueError(f"Missing downloaded price data for: {missing_tickers}")

price_output_path = save_price_data(prices, PROCESSED_DATA_DIR / "adjusted_close_prices.csv")
price_output_path

PosixPath('/Users/rovs/Documents/New project 2/projects/portfolio-optimization-factor-lab/data/processed/adjusted_close_prices.csv')

## Adjusted close price review

In [4]:
display(prices.head())
display(prices.tail())
print(f"Price data shape: {prices.shape}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")

Ticker,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
Date,,,,,,,,,
2019-01-02,37.503723,94.397141,80.836533,75.348236,50.001839,104.387886,37.582687,3.376983,224.382538
2019-01-03,33.768066,90.924477,79.687675,74.819962,49.234131,102.729126,37.350449,3.172956,219.028152
2019-01-04,35.209614,95.153297,82.625397,76.347054,51.049370,104.453262,38.095219,3.376240,226.364639
2019-01-07,35.131245,95.274658,82.682838,76.041634,51.314854,103.783234,37.598709,3.554980,228.149429
2019-01-08,35.800957,95.965462,82.526932,76.322304,51.687946,106.193764,38.023144,3.466478,230.293015


Ticker,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
Date,,,,,,,,,
2026-04-21,266.170013,424.160004,313.000000,141.256348,148.360001,226.160004,74.699997,199.880005,704.080017
2026-04-22,273.170013,432.920013,313.019989,141.782379,149.500000,226.100006,74.629997,202.500000,711.210022
2026-04-23,273.429993,415.750000,311.690002,144.621002,150.529999,230.649994,76.279999,199.639999,708.450012
2026-04-24,271.059998,424.619995,308.279999,148.179993,148.910004,227.500000,76.629997,208.270004,713.940002
2026-04-27,267.600006,420.795013,310.950012,148.789993,150.509903,226.774002,76.250000,209.460007,713.515015


Price data shape: (1839, 9)
Date range: 2019-01-02 to 2026-04-27


## Daily returns

In [5]:
daily_returns = calculate_daily_returns(prices)
daily_returns.to_csv(RETURNS_DIR / "daily_returns.csv", index_label="date")
display(daily_returns.head())
print(f"Daily returns shape: {daily_returns.shape}")

Ticker,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
Date,,,,,,,,,
2019-01-03,-0.099608,-0.036788,-0.014212,-0.007011,-0.015354,-0.015890,-0.006179,-0.060417,-0.023863
2019-01-04,0.042690,0.046509,0.036865,0.020410,0.036870,0.016783,0.019940,0.064068,0.033496
2019-01-07,-0.002226,0.001275,0.000695,-0.004000,0.005201,-0.006415,-0.013033,0.052941,0.007885
2019-01-08,0.019063,0.007251,-0.001886,0.003691,0.007271,0.023227,0.011289,-0.024895,0.009396
2019-01-09,0.016981,0.014299,-0.001690,-0.016331,0.005275,-0.007926,-0.019166,0.019667,0.004673


Daily returns shape: (1838, 9)


## Monthly returns

In [6]:
monthly_returns = calculate_monthly_returns(prices)
monthly_returns.to_csv(RETURNS_DIR / "monthly_returns.csv", index_label="date")
display(monthly_returns.head())
print(f"Monthly returns shape: {monthly_returns.shape}")

Ticker,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
Date,,,,,,,,,
2019-02-28,0.044777,0.077358,0.008309,0.021561,0.090439,0.033561,-0.057968,0.074219,0.032415
2019-03-31,0.097026,0.052754,-0.029992,0.055810,0.022397,0.023053,0.042547,0.164009,0.018100
2019-04-30,0.056436,0.107343,0.155171,0.030578,-0.006436,0.010087,0.046948,0.008020,0.040853
2019-05-31,-0.124213,-0.049481,-0.086945,-0.033527,-0.108356,-0.064820,0.001427,-0.250748,-0.063771
2019-06-30,0.130519,0.083118,0.055115,0.065494,0.082803,0.061990,0.044525,0.212387,0.069586


Monthly returns shape: (87, 9)


## Annualized risk and return summary

In [7]:
performance_summary = summarize_asset_performance(daily_returns)
performance_summary.to_csv(RETURNS_DIR / "asset_performance_summary.csv", index_label="ticker")
display(performance_summary.style.format({
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
}))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 736, in start
    se

AttributeError: _ARRAY_API not found

,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown
Ticker,,,,
NVDA,76.10%,50.96%,1.49,-66.34%
AAPL,30.92%,30.84%,1.00,-33.36%
MSFT,22.74%,28.60%,0.80,-37.15%
JPM,20.29%,29.69%,0.68,-43.63%
SPY,17.19%,19.57%,0.88,-33.72%
XOM,16.31%,31.08%,0.52,-60.35%
JNJ,11.22%,19.12%,0.59,-27.37%
KO,10.19%,19.73%,0.52,-36.99%
PG,9.78%,20.12%,0.49,-23.77%


## Correlation analysis

In [8]:
correlation_matrix = daily_returns.corr()
correlation_matrix.to_csv(RETURNS_DIR / "correlation_matrix.csv", index_label="ticker")
display(correlation_matrix.style.format("{:.2f}"))

Ticker,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA,SPY
Ticker,,,,,,,,,
AAPL,1.00,0.68,0.42,0.37,0.28,0.31,0.36,0.57,0.77
MSFT,0.68,1.00,0.42,0.35,0.22,0.28,0.33,0.65,0.79
JPM,0.42,0.42,1.00,0.31,0.52,0.34,0.43,0.35,0.71
PG,0.37,0.35,0.31,1.00,0.21,0.55,0.63,0.16,0.47
XOM,0.28,0.22,0.52,0.21,1.00,0.27,0.37,0.19,0.49
JNJ,0.31,0.28,0.34,0.55,0.27,1.00,0.52,0.09,0.43
KO,0.36,0.33,0.43,0.63,0.37,0.52,1.00,0.14,0.52
NVDA,0.57,0.65,0.35,0.16,0.19,0.09,0.14,1.00,0.70
SPY,0.77,0.79,0.71,0.47,0.49,0.43,0.52,0.70,1.00


## Drawdown analysis

In [9]:
drawdown_summary = calculate_max_drawdown(daily_returns).sort_values()
display(drawdown_summary.to_frame("max_drawdown").style.format("{:.2%}"))

,max_drawdown
Ticker,
NVDA,-66.34%
XOM,-60.35%
JPM,-43.63%
MSFT,-37.15%
KO,-36.99%
SPY,-33.72%
AAPL,-33.36%
JNJ,-27.37%
PG,-23.77%


## Benchmark comparison against SPY

In [10]:
benchmark_metrics = performance_summary.loc[[BENCHMARK]].rename(index={BENCHMARK: "Benchmark: SPY"})
asset_metrics = performance_summary.loc[TICKERS]
benchmark_comparison = asset_metrics.assign(
    excess_annualized_return=asset_metrics["annualized_return"] - performance_summary.loc[BENCHMARK, "annualized_return"],
    volatility_gap=asset_metrics["annualized_volatility"] - performance_summary.loc[BENCHMARK, "annualized_volatility"],
)
display(benchmark_metrics.style.format("{:.2%}", subset=["annualized_return", "annualized_volatility", "max_drawdown"]))
display(benchmark_comparison.sort_values("excess_annualized_return", ascending=False).style.format({
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
    "excess_annualized_return": "{:.2%}",
    "volatility_gap": "{:.2%}",
}))

,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown
Ticker,,,,
Benchmark: SPY,17.19%,19.57%,0.878123,-33.72%


,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,excess_annualized_return,volatility_gap
Ticker,,,,,,
NVDA,76.10%,50.96%,1.49,-66.34%,58.92%,31.38%
AAPL,30.92%,30.84%,1.00,-33.36%,13.73%,11.27%
MSFT,22.74%,28.60%,0.80,-37.15%,5.55%,9.03%
JPM,20.29%,29.69%,0.68,-43.63%,3.10%,10.12%
XOM,16.31%,31.08%,0.52,-60.35%,-0.88%,11.51%
JNJ,11.22%,19.12%,0.59,-27.37%,-5.96%,-0.46%
KO,10.19%,19.73%,0.52,-36.99%,-7.00%,0.16%
PG,9.78%,20.12%,0.49,-23.77%,-7.41%,0.54%


## Figures

In [11]:
figures = {
    "price_history.png": (plot_price_history(prices), "lines", prices),
    "cumulative_returns.png": (plot_cumulative_returns(daily_returns), "lines", (1 + daily_returns.fillna(0)).cumprod() - 1),
    "correlation_heatmap.png": (plot_correlation_heatmap(daily_returns), "heatmap", correlation_matrix),
    "risk_return_scatter.png": (plot_risk_return_scatter(performance_summary), "scatter", performance_summary),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figures.items():
    method = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)
    figure_export_methods[filename] = method

figure_export_methods

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



{'price_history.png': 'pillow_fallback',
 'cumulative_returns.png': 'pillow_fallback',
 'correlation_heatmap.png': 'pillow_fallback',
 'risk_return_scatter.png': 'pillow_fallback'}

## Initial business interpretation

- This universe combines large-cap technology, financials, consumer staples, energy, healthcare, beverages, and semiconductor exposure.
- SPY provides a broad U.S. equity benchmark for comparing asset-level return and risk.
- The risk-return summary, drawdown table, and correlation matrix identify which assets have historically contributed higher return, higher volatility, and higher diversification potential since 2019.
- These outputs are descriptive research inputs only and are not investment recommendations.

## Phase 1 limitations

- Yahoo Finance data can change because of vendor revisions, ticker availability, and corporate action adjustments.
- Returns are historical and do not imply future performance.
- No CAPM, factor model, portfolio optimization, transaction cost model, or backtest has been built in this phase.
- The risk-free rate is held at zero for the initial Sharpe ratio summary and will be refined later if needed.

## Next steps for Phase 2 CAPM and factor research

- Estimate benchmark beta and alpha for each asset using SPY as the market proxy.
- Add factor research inputs after the return foundation is stable.
- Compare asset behavior across market cycles and identify financially explainable drivers.